##### This file let's you calculates the quantitative metrics for any pooling method and seed.  


In [1]:
# Imports
import pandas as pd
import numpy as np
import os
import sys
import pickle
import torch
from collections import defaultdict
from scipy.stats import spearmanr
import shap
import json

In [2]:
# Add project root to path to find models.py
project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..')) \
    if '__file__' in globals() else os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from models import PolicyNetwork, create_mil_model_with_dict

In [3]:
# Main Configuration
SEED_TO_ANALYZE = 8
POOLING_METHOD_TO_ANALYZE = "repset"
TOP_K_FOR_COUNTS = 20
REPRODUCIBILITY_SEED = 42
DEMOGRAPHIC_LABELS = ['Imd Band', 'Region', 'Gender', 'Age Band', 'Disability']

OUTPUT_DIR = '../results/'


# These depend on which specific experiments you did
DATASET_CONFIGS = {
    "oulad_full": {
        "base_path": f'/projects/prjs1491/Attention-based-RL-MIL/runs/classification/seed_{SEED_TO_ANALYZE}/oulad_full/instances/tabular/label/bag_size_20/{POOLING_METHOD_TO_ANALYZE}_20_16_20',
        "raw_data_path": '/projects/prjs1491/Attention-based-RL-MIL/data/oulad/oulad_full_raw.pkl',
        "output_dir": OUTPUT_DIR,
        "models": {
            "Linear Attention": {
                "run_dir_suffix": 'neg_policy_only_loss_attention_linear_reg_sum_sample_without_replacement/',
                "output_file": "attention_linear_outputs.csv",
                "score_column": "attention_score"
            },
            "Gated Attention": {
                "run_dir_suffix": 'neg_policy_only_loss_attention_gated_reg_sum_sample_without_replacement/',
                "output_file": "attention_gated_outputs.csv",
                "score_column": "attention_score"
            },
            "Multi-Head Attention": {
                "run_dir_suffix": 'neg_policy_only_loss_attention_multi_head_reg_sum_sample_without_replacement/',
                "output_file": "attention_multi_head_outputs.csv",
                "score_column": "attention_score"
            },
            "Epsilon-Greedy (SHAP)": {
                "is_shap_model": True,
                "score_column": "shap_value",
                # Which model's weights to load for the explanation
                "model_to_explain_suffix": 'neg_policy_only_loss_epsilon_greedy_reg_sum_sample_without_replacement/',
                # Which model's output file to use for the instance features
                "feature_source_file": "attention_gated_outputs.csv",
                "feature_source_suffix": 'neg_policy_only_loss_attention_gated_reg_sum_sample_without_replacement/'
            }
        }
    },
    "oulad_aggregated": {
        "base_path": f'/projects/prjs1491/Attention-based-RL-MIL/runs/classification/seed_{SEED_TO_ANALYZE}/oulad_aggregated/instances/tabular/label/bag_size_20/{POOLING_METHOD_TO_ANALYZE}_22_16_22/',
        "raw_data_path": '/projects/prjs1491/Attention-based-RL-MIL/data/oulad/oulad_aggregated_raw.pkl',
        "output_dir": OUTPUT_DIR,
        "models": {
            "Linear Attention": {
                "run_dir_suffix": 'neg_policy_only_loss_attention_linear_reg_sum_sample_without_replacement/',
                "output_file": "attention_linear_outputs.csv",
                "score_column": "attention_score"
            },
            "Gated Attention": {
                "run_dir_suffix": 'neg_policy_only_loss_attention_gated_reg_sum_sample_without_replacement/',
                "output_file": "attention_gated_outputs.csv",
                "score_column": "attention_score"
            },
            "Multi-Head Attention": {
                "run_dir_suffix": 'neg_policy_only_loss_attention_multi_head_reg_sum_sample_without_replacement/',
                "output_file": "attention_multi_head_outputs.csv",
                "score_column": "attention_score"
            },
            "Epsilon-Greedy (SHAP)": {
                "is_shap_model": True,
                "score_column": "shap_value",
                "model_to_explain_suffix": 'neg_policy_only_loss_epsilon_greedy_reg_sum_sample_without_replacement/',
                "feature_source_file": "attention_gated_outputs.csv",
                "feature_source_suffix": 'neg_policy_only_loss_attention_gated_reg_sum_sample_without_replacement/'
            }
        }
    }
}

for config in DATASET_CONFIGS.values():
    os.makedirs(config['output_dir'], exist_ok=True)

In [4]:
# Helper functions
def generate_general_labels(raw_bag_data):
    labels = []
    for instance_list in raw_bag_data:
        main_tuple = instance_list[0]
        if main_tuple[0] == 'assessment_type':
            labels.append(f"Assessment: {main_tuple[1]}")
        elif main_tuple[0] == 'activity_type':
            labels.append(f"VLE Clicks: {main_tuple[1]}")
        else:
            labels.append(main_tuple[0].replace('_', ' ').title())
    return labels

def build_bag_to_labels_map(raw_data_path):
    print("\nBuilding global map from Bag ID to Instance Labels")
    try:
        with open(raw_data_path, 'rb') as f:
            data = pickle.load(f)
    except FileNotFoundError:
        print(f"FATAL: Raw data file not found at {raw_data_path}")
        return {}

    # Debug check
    if not all(k in data for k in ("bag_ids", "raw_bags")):
        print(f"Unexpected pickle format. Keys found: {list(data.keys())}")
        return {}

    bag_to_labels = {}
    for bag_id, raw_bag in zip(data['bag_ids'], data['raw_bags']):
        bag_to_labels[str(bag_id)] = generate_general_labels(raw_bag)

    print(f"Built map for {len(bag_to_labels)} bags.")
    return bag_to_labels


def print_counts_table(model_name, counts_dict, bag_presence_dict, total_bags):
    print(f"\n--- Top-{TOP_K_FOR_COUNTS} Feature Counts for: {model_name.upper()} ---")
    if not counts_dict:
        print("No instances were found in the top 20.")
        return

    df = pd.DataFrame(counts_dict.items(), columns=['Feature Type', 'Total Count in Top-20'])
    df['Present in X Bags'] = df['Feature Type'].map(bag_presence_dict)
    df['Bag Presence %'] = df['Present in X Bags'] / total_bags * 100
    df['Avg Count When Present'] = df['Total Count in Top-20'] / df['Present in X Bags']
    df = df.sort_values(by='Total Count in Top-20', ascending=False).reset_index(drop=True)
    print(f"Based on {total_bags} common bags.")
    display(df)


def calculate_hhi(counts_dict, total_bags):
    total_top_k_instances = total_bags * TOP_K_FOR_COUNTS
    if total_top_k_instances == 0: return 0
    frequencies = np.array(list(counts_dict.values())) / total_top_k_instances
    return np.sum(frequencies**2)


def calculate_gini(counts_dict):
    counts = np.array(list(counts_dict.values()), dtype=np.float64)
    if np.sum(counts) == 0: return 0
    sorted_counts = np.sort(counts)
    n = len(counts)
    cum_counts = np.cumsum(sorted_counts)
    return (n + 1 - 2 * np.sum(cum_counts) / cum_counts[-1]) / n


def get_counts_for_split(bags_list, df_source, score_column, bag_to_labels_map):
    counts = defaultdict(int)
    for bag_id in bags_list:
        if bag_id not in bag_to_labels_map: continue
        bag_labels = bag_to_labels_map[bag_id]
        bag_df = df_source[df_source['bag_id'] == bag_id].reset_index(drop=True)
        if len(bag_labels) != len(bag_df): continue
        top_indices = bag_df.nlargest(TOP_K_FOR_COUNTS, score_column).index
        for idx in top_indices:
            counts[bag_labels[idx]] += 1
    return counts

def load_rl_model(run_dir_path):
    print("Attempting to load RL model...")
    device = torch.device("cpu")
    model_weights_path = os.path.join(run_dir_path, 'sweep_best_model.pt')
    rl_config_path = os.path.join(run_dir_path, 'sweep_best_model_config.json')
    mil_config_path = os.path.join(run_dir_path, '..', 'best_model_config.json')
    mil_weights_path = os.path.join(run_dir_path, '..', 'best_model.pt')
    
    try:
        with open(mil_config_path) as f: mil_config = json.load(f)
        with open(rl_config_path) as f: rl_config = json.load(f)
    except FileNotFoundError as e:
        print(f"FATAL: A config file was not found: {e}"); return None

    task_model = create_mil_model_with_dict(mil_config)
    task_model.load_state_dict(torch.load(mil_weights_path, map_location=device))
    
    policy_network = PolicyNetwork(
        task_model=task_model, state_dim=rl_config['state_dim'], hdim=rl_config['hdim'],
        learning_rate=rl_config['learning_rate'], device=device, task_type=rl_config['task_type'],
        min_clip=rl_config.get('min_clip'), max_clip=rl_config.get('max_clip'),
        sample_algorithm=rl_config.get('sample_algorithm'), no_autoencoder=rl_config.get('no_autoencoder_for_rl', False)
    )
    policy_network.load_state_dict(torch.load(model_weights_path, map_location=device))
    policy_network.eval()
    print("RL model loaded successfully.")
    return policy_network

def parse_feature_string(s):
    if not isinstance(s, str): return np.array([])
    return np.fromstring(s.replace('[','').replace(']','').replace('\n',' '), sep=' ')

def append_to_csv(df, filepath):
    file_exists = os.path.exists(filepath)
    df.to_csv(filepath, mode='a', header=(not file_exists), index=False)
    
    if not file_exists:
        print(f"Created new file and saved results to: {filepath}")
    else:
        print(f"Appended new results to: {filepath}")


In [5]:
# Step 1: Load Bag-to-Label Maps

bag_to_labels_maps = {}

for dataset_name, dataset_config in DATASET_CONFIGS.items():
    print(f"--- Loading label map for: {dataset_name.upper()} ---")
    try:
        bag_to_labels_maps[dataset_name] = build_bag_to_labels_map(dataset_config['raw_data_path'])
        print(f"Successfully loaded map with {len(bag_to_labels_maps[dataset_name])} bags.")
    except Exception as e:
        print(f"    ERROR: Could not load map for {dataset_name}. Error: {e}")

--- Loading label map for: OULAD_FULL ---

Building global map from Bag ID to Instance Labels
Built map for 27379 bags.
Successfully loaded map with 27379 bags.
--- Loading label map for: OULAD_AGGREGATED ---

Building global map from Bag ID to Instance Labels
Built map for 27379 bags.
Successfully loaded map with 27379 bags.


In [6]:
# Step 2: Calculate or Load SHAP Values

all_dfs = {} 

print("Starting SHAP Analysis")

for dataset_name, dataset_config in DATASET_CONFIGS.items():
    for model_name, model_config in dataset_config['models'].items():
        
        # Process only if the model is explicitly marked for SHAP analysis
        if model_config.get("is_shap_model"):
            full_model_name = f"{model_name} on {dataset_name}"
            print(f"\n- Processing SHAP for: {full_model_name}")
            
            shap_filename = f"shap_scores_{dataset_name}_{POOLING_METHOD_TO_ANALYZE}_seed_{SEED_TO_ANALYZE}.csv"
            shap_output_file = os.path.join(dataset_config['output_dir'], shap_filename)
            
            try:
                feature_path = os.path.join(dataset_config['base_path'], model_config['feature_source_suffix'], model_config['feature_source_file'])
                source_df = pd.read_csv(feature_path)
                source_df = source_df[source_df['is_padding_instance'] == False]

                if os.path.exists(shap_output_file):
                    print(f"  Found pre-computed file. Loading from: {shap_output_file}")
                    shap_df_loaded = pd.read_csv(shap_output_file, index_col=0)
                    df = source_df.join(shap_df_loaded)
                else:
                    # SHAP CALCULATION LOGIC
                    print(f"  No SHAP file found at {shap_output_file}.")
                    print("  Starting new calculation (this may take a while)...")
                    
                    model_dir = os.path.join(dataset_config['base_path'], model_config['model_to_explain_suffix'])
                    greedy_model = load_rl_model(model_dir) # Assumes load_rl_model is defined
                    
                    if not greedy_model:
                        print(f"  ERROR: Could not load model from {model_dir}. Skipping.")
                        continue

                    instance_features = np.vstack(source_df['original_instance_content'].apply(parse_feature_string))
                    
                    def shap_wrapper_f(x):
                        with torch.no_grad():
                            return greedy_model(torch.from_numpy(x).float().to("cpu"))[0].cpu().numpy()
                    
                    explainer = shap.KernelExplainer(shap_wrapper_f, shap.sample(instance_features, 50))
                    shap_values = explainer.shap_values(instance_features)
                    
                    shap_df = pd.DataFrame({'shap_value': np.abs(shap_values).mean(axis=1)}, index=source_df.index)
                    
                    print(f"  SHAP values calculated. Saving to: {shap_output_file}")
                    shap_df.to_csv(shap_output_file)
                    
                    df = source_df.join(shap_df)

                all_dfs[full_model_name] = df

            except (FileNotFoundError, KeyError) as e:
                print(f"  WARNING: Could not process {full_model_name}. Error: {e}")

print("\n SHAP analysis complete.")

--- Starting SHAP Analysis ---

- Processing SHAP for: Epsilon-Greedy (SHAP) on oulad_full
  Found pre-computed file. Loading from: ../results/shap_scores_oulad_full_repset_seed_8.csv

- Processing SHAP for: Epsilon-Greedy (SHAP) on oulad_aggregated
  Found pre-computed file. Loading from: ../results/shap_scores_oulad_aggregated_repset_seed_8.csv

--- SHAP analysis complete. ---


In [7]:
# Step 3: Load Standard Model Data

print("Loading standard model data")

for dataset_name, dataset_config in DATASET_CONFIGS.items():
    for model_name, model_config in dataset_config['models'].items():
        
        if not model_config.get("is_shap_model"):
            full_model_name = f"{model_name} on {dataset_name}"
            print(f"Loading: {full_model_name}")
            
            try:
                model_path = os.path.join(dataset_config['base_path'], model_config['run_dir_suffix'], model_config['output_file'])
                df = pd.read_csv(model_path)
                df = df[df['is_padding_instance'] == False]
                
                all_dfs[full_model_name] = df
            except (FileNotFoundError, KeyError) as e:
                print(f"  WARNING: Could not process {full_model_name}. Error: {e}")

--- Loading standard model data ---
Loading: Linear Attention on oulad_full
Loading: Gated Attention on oulad_full
Loading: Multi-Head Attention on oulad_full
Loading: Linear Attention on oulad_aggregated
Loading: Gated Attention on oulad_aggregated
Loading: Multi-Head Attention on oulad_aggregated

--- Standard model loading complete. ---


In [8]:
# Step 4: Find Common Bags Per Dataset

common_bags_per_dataset = {}
dfs_by_dataset = {dataset_name: {} for dataset_name in DATASET_CONFIGS.keys()}

for model_name, df in all_dfs.items():
    for dataset_name in DATASET_CONFIGS.keys():
        if dataset_name in model_name:
            dfs_by_dataset[dataset_name][model_name] = df
            break

for dataset_name, model_dfs in dfs_by_dataset.items():
    if not model_dfs:
        print(f"No models loaded for dataset '{dataset_name}'. Skipping.")
        continue

    common_bags = set.intersection(*(set(df['bag_id'].unique()) for df in model_dfs.values()))
    common_bags_per_dataset[dataset_name] = sorted(list(common_bags))
    print(f"Found {len(common_bags_per_dataset[dataset_name])} common bags for the {len(model_dfs)} models in '{dataset_name}'.")

--- Finding common bags for each dataset ---
- Found 2738 common bags for the 4 models in 'oulad_full'.
- Found 2738 common bags for the 4 models in 'oulad_aggregated'.


In [9]:
# Step 5: Generate Top-20 Counts

all_counts = {name: defaultdict(int) for name in all_dfs.keys()}
all_bag_presence = {name: defaultdict(int) for name in all_dfs.keys()}

for model_name, df_source in all_dfs.items():
    print(f"Processing: {model_name}")
    
    dataset_name = "oulad_full" if "oulad_full" in model_name else "oulad_aggregated"
    correct_map = bag_to_labels_maps[dataset_name]
    common_bags_for_model = common_bags_per_dataset[dataset_name]
    
    base_model_name = model_name.replace(f" on {dataset_name}", "")
    score_col = DATASET_CONFIGS[dataset_name]['models'][base_model_name]['score_column']

    grouped_df = df_source.groupby('bag_id')

    for bag_id in common_bags_for_model:
        if bag_id not in correct_map:
            continue
        
        try:
            bag_df = grouped_df.get_group(bag_id)
            bag_labels = correct_map[bag_id]

            if len(bag_labels) == len(bag_df):
                unique_labels_in_top20 = set()
                top_20 = bag_df.nlargest(TOP_K_FOR_COUNTS, score_col)
                
                for idx in top_20.index:
                    label = bag_labels[bag_df.index.get_loc(idx)]
                    all_counts[model_name][label] += 1
                    unique_labels_in_top20.add(label)
                    
                for label in unique_labels_in_top20:
                    all_bag_presence[model_name][label] += 1
        except KeyError:
            continue


--- Generating Top-20 counts ---
- Processing: Epsilon-Greedy (SHAP) on oulad_full


- Processing: Epsilon-Greedy (SHAP) on oulad_aggregated
- Processing: Linear Attention on oulad_full
- Processing: Gated Attention on oulad_full
- Processing: Multi-Head Attention on oulad_full
- Processing: Linear Attention on oulad_aggregated
- Processing: Gated Attention on oulad_aggregated
- Processing: Multi-Head Attention on oulad_aggregated

--- Count generation complete. ---


In [10]:
# Step 6: Print Descriptive Statistics

for model_name in all_dfs.keys():
    
    if "oulad_full" in model_name:
        dataset_name = "oulad_full"
    elif "oulad_aggregated" in model_name:
        dataset_name = "oulad_aggregated"
    else:
        continue

    total_bags_for_dataset = len(common_bags_per_dataset[dataset_name])
    
    print_counts_table(
        model_name, 
        all_counts[model_name], 
        all_bag_presence[model_name], 
        total_bags_for_dataset
    )


--- Top-20 Feature Counts for: EPSILON-GREEDY (SHAP) ON OULAD_FULL ---
Based on 2738 common bags.


,Feature Type,Total Count in Top-20,Present in X Bags,Bag Presence %,Avg Count When Present
0,VLE Clicks: homepage,11780,2506,91.526662,4.700718
1,VLE Clicks: subpage,5521,1994,72.826881,2.768806
2,Assessment: TMA,4679,1909,69.722425,2.451021
3,VLE Clicks: forumng,4667,1641,59.934259,2.843998
4,VLE Clicks: oucontent,3759,1427,52.118335,2.634198
5,Assessment: CMA,3446,1191,43.498904,2.893367
6,VLE Clicks: resource,3031,1593,58.181154,1.902699
7,VLE Clicks: quiz,1508,869,31.738495,1.735328
8,VLE Clicks: url,1486,811,29.620161,1.832306
9,Num Of Prev Attempts,1145,1145,41.818846,1.000000



--- Top-20 Feature Counts for: EPSILON-GREEDY (SHAP) ON OULAD_AGGREGATED ---
Based on 2738 common bags.


,Feature Type,Total Count in Top-20,Present in X Bags,Bag Presence %,Avg Count When Present
0,Assessment: TMA,9719,2415,88.203068,4.024431
1,Assessment: CMA,5043,1424,52.008766,3.541433
2,VLE Clicks: homepage,2683,2683,97.991234,1.000000
3,VLE Clicks: subpage,2619,2619,95.653762,1.000000
4,VLE Clicks: resource,2591,2591,94.631118,1.000000
5,VLE Clicks: oucontent,2500,2500,91.307524,1.000000
6,VLE Clicks: forumng,2470,2470,90.211833,1.000000
7,Code Presentation,2323,2323,84.842951,1.000000
8,VLE Clicks: url,2259,2259,82.505478,1.000000
9,Highest Education,2126,2126,77.647918,1.000000



--- Top-20 Feature Counts for: LINEAR ATTENTION ON OULAD_FULL ---
Based on 2738 common bags.


...

,Feature Type,Total Count in Top-20,Present in X Bags,Bag Presence %,Avg Count When Present
0,Assessment: TMA,9755,2415,88.203068,4.039337
1,Assessment: CMA,4829,1409,51.460920,3.427253
2,VLE Clicks: homepage,2683,2683,97.991234,1.000000
3,VLE Clicks: subpage,2615,2615,95.507670,1.000000
4,VLE Clicks: resource,2585,2585,94.411980,1.000000
5,VLE Clicks: oucontent,2505,2505,91.490139,1.000000
6,VLE Clicks: forumng,2480,2480,90.577064,1.000000
7,Code Module,2357,2357,86.084733,1.000000
8,Imd Band,2287,2287,83.528123,1.000000
9,VLE Clicks: url,2172,2172,79.327977,1.000000



--- Top-20 Feature Counts for: GATED ATTENTION ON OULAD_AGGREGATED ---
Based on 2738 common bags.


,Feature Type,Total Count in Top-20,Present in X Bags,Bag Presence %,Avg Count When Present
0,Assessment: CMA,5313,1434,52.373996,3.705021
1,Assessment: TMA,3972,1615,58.984660,2.459443
2,Highest Education,2738,2738,100.000000,1.000000
3,Imd Band,2728,2728,99.634770,1.000000
4,VLE Clicks: homepage,2683,2683,97.991234,1.000000
5,VLE Clicks: subpage,2619,2619,95.653762,1.000000
6,Code Module,2609,2609,95.288532,1.000000
7,VLE Clicks: resource,2591,2591,94.631118,1.000000
8,VLE Clicks: oucontent,2507,2507,91.563185,1.000000
9,Date Registration,2489,2489,90.905771,1.000000



--- Top-20 Feature Counts for: MULTI-HEAD ATTENTION ON OULAD_AGGREGATED ---
Based on 2738 common bags.


,Feature Type,Total Count in Top-20,Present in X Bags,Bag Presence %,Avg Count When Present
0,Assessment: TMA,3158,1362,49.744339,2.318649
1,Assessment: CMA,2917,1066,38.933528,2.736398
2,Age Band,2668,2668,97.443389,1.000000
3,Studied Credits,2660,2660,97.151205,1.000000
4,Disability,2646,2646,96.639883,1.000000
5,Num Of Prev Attempts,2608,2608,95.252009,1.000000
6,Region,2553,2553,93.243243,1.000000
7,Date Registration,2526,2526,92.257122,1.000000
8,Gender,2519,2519,92.001461,1.000000
9,Highest Education,2468,2468,90.138787,1.000000


In [11]:
# Step 7: Reliance on Demographics

reliance_results = []
for model_name in all_dfs.keys():
    if "oulad_full" in model_name:
        dataset_name = "oulad_full"
    elif "oulad_aggregated" in model_name:
        dataset_name = "oulad_aggregated"
    else:
        continue

    total_possible_instances = len(common_bags_per_dataset[dataset_name]) * TOP_K_FOR_COUNTS
    demographic_instance_count = sum(
        all_counts[model_name].get(label, 0) for label in DEMOGRAPHIC_LABELS
    )
    if total_possible_instances > 0:
        reliance_percentage = (demographic_instance_count / total_possible_instances) * 100
    else:
        reliance_percentage = 0
        
    reliance_results.append({
        "Model": model_name,
        "Reliance on Demographics (%)": f"{reliance_percentage:.1f}%"
    })

reliance_df = pd.DataFrame(reliance_results)

display(reliance_df.sort_values(by="Model").reset_index(drop=True))

--- Reliance on Demographics ---


,Model,Reliance on Demographics (%)
0,Epsilon-Greedy (SHAP) on oulad_aggregated,13.1%
1,Epsilon-Greedy (SHAP) on oulad_full,8.8%
2,Gated Attention on oulad_aggregated,15.6%
3,Gated Attention on oulad_full,6.5%
4,Linear Attention on oulad_aggregated,11.1%
5,Linear Attention on oulad_full,5.7%
6,Multi-Head Attention on oulad_aggregated,23.3%
7,Multi-Head Attention on oulad_full,16.8%


In [12]:
# Step 8: Sparsity Metrics
sparsity_results = []
for model_name in all_dfs.keys():
    
    if "oulad_full" in model_name:
        dataset_name = "oulad_full"
    elif "oulad_aggregated" in model_name:
        dataset_name = "oulad_aggregated"
    else:
        continue
        
    total_bags_for_dataset = len(common_bags_per_dataset[dataset_name])
    sparsity_results.append({
        "Model": model_name,
        "Gini Coefficient": calculate_gini(all_counts[model_name]),
        "HHI Score": calculate_hhi(all_counts[model_name], total_bags_for_dataset),
    })

sparsity_df = pd.DataFrame(sparsity_results)

display(sparsity_df.sort_values(by="Model").reset_index(drop=True))


--- Sparsity Metrics ---


,Model,Gini Coefficient,HHI Score
0,Epsilon-Greedy (SHAP) on oulad_aggregated,0.503193,0.064211
1,Epsilon-Greedy (SHAP) on oulad_full,0.608344,0.088632
2,Gated Attention on oulad_aggregated,0.422240,0.044976
3,Gated Attention on oulad_full,0.730198,0.147605
4,Linear Attention on oulad_aggregated,0.493194,0.063529
5,Linear Attention on oulad_full,0.702741,0.136693
6,Multi-Head Attention on oulad_aggregated,0.364944,0.040062
7,Multi-Head Attention on oulad_full,0.576559,0.075561


In [13]:
# Step 9: Consistency Metrics (Split-Half Method) 

consistency_results = []
np.random.seed(REPRODUCIBILITY_SEED)

for dataset_name, common_bags in common_bags_per_dataset.items():
    
    print(f"\n- Splitting bags for dataset: {dataset_name}")
    
    shuffled_bags = np.array(common_bags)
    np.random.shuffle(shuffled_bags)
    split_point = len(shuffled_bags) // 2
    bags_a, bags_b = shuffled_bags[:split_point], shuffled_bags[split_point:]
    
    correct_map_for_model = bag_to_labels_maps[dataset_name]

    for model_name, df in all_dfs.items():
        if dataset_name in model_name:
            print(f"  -> Calculating for model: {model_name}...")
            
            base_model_name = model_name.replace(f" on {dataset_name}", "")
            score_col = DATASET_CONFIGS[dataset_name]['models'][base_model_name]['score_column']
            
            counts_a = get_counts_for_split(bags_a, df, score_col, correct_map_for_model)
            counts_b = get_counts_for_split(bags_b, df, score_col, correct_map_for_model)
            
            policy_df = pd.DataFrame({'split_A': counts_a, 'split_B': counts_b}).fillna(0)
            spearman_corr, _ = spearmanr(policy_df['split_A'], policy_df['split_B'])
            
            top5_a = set(policy_df.nlargest(5, 'split_A').index)
            top5_b = set(policy_df.nlargest(5, 'split_B').index)

            print(f"    - Top 5 (Split A): {sorted(list(top5_a))}")
            print(f"    - Top 5 (Split B): {sorted(list(top5_b))}")
            
            jaccard_k5_score = len(top5_a.intersection(top5_b)) / len(top5_a.union(top5_b)) if top5_a and top5_b else 0.0

            result_row = {"Model": model_name, "Spearman's Rank (Stability)": spearman_corr, "Jaccard (K=5)": jaccard_k5_score}
            consistency_results.append(result_row)

consistency_df = pd.DataFrame(consistency_results)
display(consistency_df.sort_values(by="Model").reset_index(drop=True))

--- Calculating Consistency Metrics ---

- Splitting bags for dataset: oulad_full
  -> Calculating for model: Epsilon-Greedy (SHAP) on oulad_full...


    - Top 5 (Split A): ['Assessment: TMA', 'VLE Clicks: forumng', 'VLE Clicks: homepage', 'VLE Clicks: oucontent', 'VLE Clicks: subpage']
    - Top 5 (Split B): ['Assessment: TMA', 'VLE Clicks: forumng', 'VLE Clicks: homepage', 'VLE Clicks: oucontent', 'VLE Clicks: subpage']
  -> Calculating for model: Linear Attention on oulad_full...
    - Top 5 (Split A): ['Assessment: CMA', 'Assessment: TMA', 'VLE Clicks: forumng', 'VLE Clicks: homepage', 'VLE Clicks: oucontent']
    - Top 5 (Split B): ['Assessment: CMA', 'Assessment: TMA', 'VLE Clicks: forumng', 'VLE Clicks: homepage', 'VLE Clicks: oucontent']
  -> Calculating for model: Gated Attention on oulad_full...
    - Top 5 (Split A): ['VLE Clicks: forumng', 'VLE Clicks: homepage', 'VLE Clicks: oucontent', 'VLE Clicks: resource', 'VLE Clicks: subpage']
    - Top 5 (Split B): ['VLE Clicks: forumng', 'VLE Clicks: homepage', 'VLE Clicks: oucontent', 'VLE Clicks: resource', 'VLE Clicks: subpage']
  -> Calculating for model: Multi-Head Attentio

,Model,Spearman's Rank (Stability),Jaccard (K=5)
0,Epsilon-Greedy (SHAP) on oulad_aggregated,0.997402,1.0
1,Epsilon-Greedy (SHAP) on oulad_full,0.997242,1.0
2,Gated Attention on oulad_aggregated,0.999389,1.0
3,Gated Attention on oulad_full,0.994234,1.0
4,Linear Attention on oulad_aggregated,0.997861,1.0
5,Linear Attention on oulad_full,0.986379,1.0
6,Multi-Head Attention on oulad_aggregated,0.997555,1.0
7,Multi-Head Attention on oulad_full,0.998663,1.0


In [14]:
# Step 10: save results to csv's
SPARSITY_FILE = os.path.join(OUTPUT_DIR, 'master_sparsity_metrics.csv')
CONSISTENCY_FILE = os.path.join(OUTPUT_DIR, 'master_consistency_metrics.csv')
DEMOGRAPHICS_FILE = os.path.join(OUTPUT_DIR, 'master_demographics_metrics.csv')

sparsity_df['seed'] = SEED_TO_ANALYZE
sparsity_df['pooling_method'] = POOLING_METHOD_TO_ANALYZE

consistency_df['seed'] = SEED_TO_ANALYZE
consistency_df['pooling_method'] = POOLING_METHOD_TO_ANALYZE

reliance_df['seed'] = SEED_TO_ANALYZE
reliance_df['pooling_method'] = POOLING_METHOD_TO_ANALYZE

try:
    append_to_csv(sparsity_df, SPARSITY_FILE)
    append_to_csv(consistency_df, CONSISTENCY_FILE)
    append_to_csv(reliance_df, DEMOGRAPHICS_FILE)
except NameError as e:
    print(f"ERROR: A results DataFrame is not defined. Please ensure the notebook has run successfully. Details: {e}")

Appended new results to: ../results/master_sparsity_metrics.csv
Appended new results to: ../results/master_consistency_metrics.csv
Appended new results to: ../results/master_demographics_metrics.csv
\n--- All results for this run have been saved successfully. ---
